#### Libraries for this entire Notebook

In [0]:
# (1) FOR PARAMETER READING FROM CONFIG FILES
import yaml

# (2) FOR AUTOLOADER => RUN_ID
import uuid

# (3) FOR AUTOLOADER => TIMESTAMP
from datetime import datetime, timezone
from pyspark.sql import functions as F


#### Logging of started notebook:


In [0]:
# NOTEBOOK NAME - Get notebook path using dbutils (works on serverless compute)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

# Extract just the notebook name
notebook_name = notebook_path.split("/")[-1]

# NOTEBOOK START TIME - Current timestamp
notebook_start = datetime.now(timezone.utc).isoformat()

# LOGGING (kind-of): Notebook runtime environment - started notebook!
print(f"Notebook started with = notebook_path: {notebook_path}")
print(f"Notebook started with = notebook_name: {notebook_name}")
print(f"Notebook started with = notebook_start: {notebook_start}")

#### Project specific definitions for reusable PARAMETERS:


##### (1) Parameters: UNITY CATALOG objects

In [0]:
config_uc_objects__path = "/Workspace/Repos/michael.c.feiertag@web.de/PRJ__Vattenfall_EnergyOps_MarketIntelligence/config/config_uc_objects.yml"

with open(config_uc_objects__path, "r") as conf_uc:
    config_uc_objects = yaml.safe_load(conf_uc)

catalog_name = config_uc_objects["catalog"]
raw_schema = config_uc_objects["schemas"]["raw"]
landing_volume = config_uc_objects["volumes"]["landing"]
checkpoint_volume = config_uc_objects["volumes"]["checkpoints"]

# LOGGING (kind-of): all => used Parameters for entire Notebook runtime environment!
print(f"Used Config file: {config_uc_objects__path}")
print(f"Notebook`s Runtime Environment = catalog_name: {catalog_name}")
print(f"Notebook`s Runtime Environment = raw_schema: {raw_schema}")
print(f"Notebook`s Runtime Environment = landing_volume: {landing_volume}")
print(f"Notebook`s Runtime Environment = checkpoint_volume: {checkpoint_volume}")

##### (2) Parameters: UNITY CATALOG folders under volumes

In [0]:
config_uc_folders__path = "/Workspace/Repos/michael.c.feiertag@web.de/PRJ__Vattenfall_EnergyOps_MarketIntelligence/config/config_pathes.yml"

with open(config_uc_folders__path, "r") as conf_folders:
    config_uc_folders = yaml.safe_load(conf_folders)


# MARKET_PRICES:
landing_folder__market_prices = config_uc_folders["landing_paths"]["market_prices"]
checkpoint_folder__market_prices = config_uc_folders["checkpoint_paths"]["market_prices"]
schema_tracking_folder__market_prices = config_uc_folders["schema_tracking_paths"]["market_prices"]

landing_path__market_prices = f"/Volumes/{catalog_name}/{raw_schema}/{landing_volume}/{landing_folder__market_prices}"
checkpoint_path__market_prices = f"/Volumes/{catalog_name}/{raw_schema}/{checkpoint_volume}/{checkpoint_folder__market_prices}"
schema_tracking_path__market_prices = f"/Volumes/{catalog_name}/{raw_schema}/{checkpoint_volume}/{schema_tracking_folder__market_prices}"

# LOGGING (kind-of): all => used Parameters for entire Notebook runtime environment!
print(f"Used Config file: {config_uc_folders__path}")
print(f"NBs Rt.Env.= landing_path__market_prices: {landing_path__market_prices}")
print(f"NBs Rt.Env.= checkpoint_path__market_prices: {checkpoint_path__market_prices}")
print(f"NBs Rt.Env.= schema_tracking_path__market_prices: {schema_tracking_path__market_prices}")

##### (3) Parameters: UNITY CATALOG table under schema

In [0]:
config_uc_tables__path = "/Workspace/Repos/michael.c.feiertag@web.de/PRJ__Vattenfall_EnergyOps_MarketIntelligence/config/config_tables.yml"

with open(config_uc_tables__path, "r") as conf_tables:
    config_uc_tables = yaml.safe_load(conf_tables)

# MARKET_PRICES:
bronze_table_name = config_uc_tables["bronze_tables"]["market_prices"]

# PARAMETER: ENTIRE PATH FOR TARGET TABLE NEEDED:
# - PARAMETERS (1) catalog_name, (2) raw_schema, BOTH determined in previous CELL.
# - PARAMETER (3) bronze_table_name determined above.
bronze_table = f"{catalog_name}.{raw_schema}.{bronze_table_name}"

# LOGGING (kind-of): all => used Parameters for entire Notebook runtime environment!
print(f"Used Config file: {config_uc_tables__path}")
print(f"Notebook`s Runtime Environment = bronze_table: {bronze_table}")

#### Copying RAW files from REPO to UNITY CATALOG folder

##### Parameters: SOURCE and TARGET pathes

In [0]:
# SOURCE: Fixed from Project REPO => "sample_data"
source_path = "/Workspace/Repos/michael.c.feiertag@web.de/PRJ__Vattenfall_EnergyOps_MarketIntelligence/sample_data/market_prices"

# SOURCE: File name check
filename_pattern = "market_prices"

# TARGET: From above determination of Parameters:
landing_path = landing_path__market_prices

# LOGGING (kind-of): all => used Parameters for next Cell!
print(f"Copy Files = source_path: {source_path}")
print(f"Copy Files = filename_pattern: {filename_pattern}")
print(f"Copy Files = landing_path: {landing_path}")

##### Copy files from SOURCE path to TARGET path

In [0]:
files = dbutils.fs.ls(source_path)

for file in files:
    if file.name.startswith(filename_pattern):
        dbutils.fs.cp(file.path, f"{landing_path}/{file.name}")
        print(f"Copy Files = File `{file.name}` copied to `{landing_path}/{file.name}`.")
    else:
        print(f"Copy Files = File `{file.name}` does not match the pattern to begin with `{filename_pattern}.`")

#### DELTA AUTO-LOADER

##### (1) AUTO-LOADER: DataFrame as SOURCE

###### (1.1) AUTO-LOADER: DataFrame Parameters: 
[1] As SOURCE: landing_path. [2] As CHECKPOINT for SCHEMA_TRACKING: schema_tracking_path

In [0]:
# SOURCE = From above determination of Parameters - already used in above Cell "Copy Files":
landing_path = landing_path

# CHECKPOINT FOR SCHEMA_TRACKING = From above determination of Parameters:
schema_tracking_path = schema_tracking_path__market_prices

# LOGGING (kind-of): all => used Parameters for next Cell!
print(f"AUTO-LOADER DF = landing_path: {landing_path}")
print(f"AUTO-LOADER DF = schema_tracking_path: {schema_tracking_path}")

###### (1.2) AUTO-LOADER: DataFrame creation
NOTES: 

(1) `.format("cloudFiles")` defines to use the AUTO-LOADER method. 

(2) See if the INFERRED SCHEMA from the ingested source as of this DataFrame gets an enhanced DataFrame On-Top, which is then used in the AUTO-LOADER writer (in this solution: the AUTO-LOADER: Query)! 

In [0]:
# STANDARD CREATION OF THE DATAFRAME TO INGEST SCHEMA FROM SOURCE-FILES
bronze_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", schema_tracking_path)
    .load(landing_path)
)

###### (1.3) AUTO-LOADER: DataFrame enhancement
####### PURPOSE:

This cell defines an enhanced DataFrame On-Top of the ingested schema from Source-Files. The PURPOSE is here ONLY for enhanced LOGGING of the Query Run and the ingested Source-Files in this Query Run. 

####### ENHANCEMENTS:

(1) ADD a new column for RUN-ID.

(2) ADD a new column for TIMESTAMP of ingestion.

(3) ADD a new column for FILENAME (with source path).

- The chosen funtionionality is: 
`.withColumn("ingest_source_file", F.col("_metadata.file_path")`".

- This solution respects the  CONSTRAINT here that the alternative function "`F.input-file-name`" is NOT ALLOWED in context of files managed by DATABRICKS UNITY CATALOG! 

In [0]:
# FOR ENHANCEMENT (1): DETERMINE A UNIQUE RUN-ID TO WRITE INTO BRONZE TABLE
ingest_run_id = str(uuid.uuid4())
print (f"Preparing to write with column 'ingest_run_id': {ingest_run_id},")

# FOR ENHANCEMENT (2): FREEZE CURRENT TIMESTAMP TO WRITE INTO BRONZE TABLE
ingest_run_ts = datetime.now(timezone.utc)
print (f"Preparing to write with column 'ingest_timestamp': {ingest_run_ts}.")

# FOR ENHANCEMENT (3): DETERMINE FILENAME TO WRITE INTO BRONZE TABLE
# - get filename directly from metadata during query-execution-streaming
# = bronze_stream_df = bronze_stream_df.withColumn("ingest_source_file", F.col("_metadata.file_path"))

# ENHANCEMENT OF THE DATAFRAME ON-TOP OF PREVIOUS INGEST SCHEMA FROM SOURCE-FILES
bronze_stream_enh_df = (
    bronze_stream_df
    .withColumn("ingest_run_id", F.lit(ingest_run_id))
    .withColumn("ingest_timestamp", F.lit(ingest_run_ts).cast("timestamp"))
    .withColumn("ingest_source_file", F.col("_metadata.file_path"))
)

##### (2) AUTO-LOADER: Query as TARGET

###### (2.1) AUTO-LOADER: Query Parameters: 
[1] As TARGET: bronze_table. [2] As CHECKPOINT for SOURCE_TRACKING: checkpoint_path

In [0]:
bronze_table = bronze_table
checkpoint_path = checkpoint_path__market_prices

# LOGGING (kind-of): all => used Parameters for next Cell!
print(f"AUTO-LOADER QUERY = bronze_table: {bronze_table}")
print(f"AUTO-LOADER QUERY = checkpoint_path: {checkpoint_path}")

###### (2.2) AUTO-LOADER: Query Execution
NOTES: 

(1) `.format("delta")` and `.outputMode("append")` define to use the AUTO-LOADER method with DELTA. 

(2) `.withColumn("ingest_timestamp", F.current_timestamp())` is allowed to be added in the Query. 

This **would** makes sense to capture the timestamp really when the AUTO-LOADER Query starts to write into target. **But** when requiring that timestamp for later review which data records the query was writing, it is not easy (if not impossible!) to retrieve that timestamp from the actual query execution. Because that functionality "`F.current_timestamp())`" is a Spark SQL column EXPRESSION and not an actual timestamp value.

So, the workaround is to freeze one timestamp before the "AUTO-LOADER Query write" and use exactly that frozen timestamp value via the Query writing into the Bronze table. Therefore, see above Cell the "DataFrame enhancement" programming.

(3) See if the INFERRED SCHEMA from the ingested source as of this DataFrame gets an enhanced DataFrame On-Top, which is then used in the AUTO-LOADER writer (in this solution: HERE the AUTO-LOADER: Query)! 

In [0]:
# STANDARD QUERY DEFINITION AND QUERY EXECUTION-STREAMING
# - NOTE THAT THE USED DATAFRAME FROM INGESTED FILE SCHEMA CAN BE ENHANCED, 
#   SEE PREVIOUS CELL MARKED IN CODE WITH "ENHANCEMENT OF THE DATAFRAME ON-TOP ..."!
query = (
    bronze_stream_enh_df
    .writeStream
    .format("delta")
    .option("mergeSchema", "true")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

# QUERY EXECUTION-STREAMING: END CONDITION
query.awaitTermination()

# LOGGING (kind-of): VERIFICATION OF QUERY EXECUTION-STREAMING SUCCESS
print (f"Bronze ingestion finished writing into table `{bronze_table}`")
print (f"writing with column 'ingest_run_id': {ingest_run_id},")
print (f"writing with column 'ingest_timestamp': {ingest_run_ts}.")

##### (3) AUTO-LOADER: success verification against TARGET


In [0]:
# TEMPORARY SETTING FOR EXPERIMENTATION
bronze_table = "PRJ__vattenfall_01.raw.bronze_market_prices"
ingest_run_ts = "2026-05-08 17:09:38.612332+00:00"

# LOGGING
print (f"Starting to read from table `{bronze_table}`")
print (f"reading with column `ingest_timestamp`: {ingest_run_ts}.")

# ALTERNATIVE A - Planely showing the records from the above query-execution-streaming
#REM# --- Begin-Of-COMMENTING-OUT:
#REM# # READ FROM BRONZE TABLE, FILTERED BY TIMESTAMP
#REM# bronze_ingested_df = (
#REM#     spark.table(bronze_table)
#REM#     .filter(F.col("ingest_timestamp") == F.lit(ingest_run_ts).cast("timestamp"))
#REM# )
#REM# 
#REM# display(bronze_ingested_df)
#REM# --- End-Of-COMMENTING-OUT!


# ALTERNATIVE B - Counting the records from overall-loads and from the above query-execution-streaming
bronze_df = spark.table(bronze_table)

bronze_total_count = bronze_df.count()

#REM# # OPTIMIZATION - Old coding works but comment out, BEGIN:
#REM# bronze_ingested_df = (
#REM#     bronze_df
#REM#     .filter(F.col("ingest_timestamp") == F.lit(ingest_run_ts).cast("timestamp"))
#REM# )
#REM# 
#REM# bronze_ingested_count = bronze_ingested_df.count()
#REM# # OPTIMIZATION - Old coding works but comment out, END!

# OPTIMIZATION - New coding works, BEGIN:
bronze_ingested_count = bronze_df.filter(F.col("ingest_timestamp") == F.lit(ingest_run_ts).cast("timestamp")).count()
# OPTIMIZATION - New coding works, END!

print (f"Total records in bronze table: {bronze_total_count}")
print (f"Records ingested in bronze table: {bronze_ingested_count}")    

display(bronze_df)

### # ALTERNATIVE
### ! display(bronze_df.filter(F.col("ingest_timestamp") == F.lit(ingest_run_ts).cast("timestamp"))

#### End-Of this Notebook